# Train every NFLELO model, end to end

A personal notebook: run it any time to refresh the nflverse data and retrain **every** model in the project
on the development seasons, then fit this season's production models and print this week's predictions.
The walkthrough (`00_nflelo_walkthrough.ipynb`) *loads* saved results. This one *trains*.

**How to run** (from the repo root; the `.venv` has every dependency, including PyTorch):

```
# one-time setup (Python 3.13; torch has no wheels for 3.15 yet)
uv venv --python 3.13 .venv && uv pip install --python .venv/bin/python -r requirements-m6.txt
brew install libomp                      # macOS only, needed by LightGBM

# run it (writes the outputs back into this file)
TRAIN_MODE=full  .venv/bin/jupyter nbconvert --to notebook --execute --inplace notebooks/train_all_models.ipynb
TRAIN_MODE=quick .venv/bin/jupyter nbconvert --to notebook --execute --inplace notebooks/train_all_models.ipynb
# or:  scripts/train_all.sh [quick|full]
```

**What it will not do.** The 2020-2025 seasons are a spent holdout. Nothing here scores them again: every model
is trained and scored on its development seasons, and the locked holdout results are only *loaded* from
`experiments/runs/` and shown for context. Nothing is written to the run registry, the live ledger, the saved
fourth-down engine, `site/data/`, `outputs/` or `data/games.csv`: run records are redirected to a gitignored folder
(`data/raw/ml/_train_all_runs/`), and the last cell checks that git sees no new changes.

**Quick vs full.** `quick` skips the slow hyper-parameter searches (they re-find the constants already fixed in
the code), the 18-minute playoff-simulation backtest, and most of the fourth-down bootstrap; it retrains every
model with the settings already in the code. `full` also reruns the searches and uses the original bootstrap size, so
every "matches the record" table is comparable cell for cell.

**Saved outputs.** The outputs stored in this file come from a `full` run (73 minutes wall time on an Apple-silicon Mac; the fourth-down bootstrap alone is 37 minutes of it). `quick` takes about 25 minutes.

In [1]:
import os

# ---- configuration ----------------------------------------------------------------------------------
# TRAIN_MODE and REFRESH_DATA can be set from the shell (see above); edit the defaults here if you prefer.
MODE = os.environ.get("TRAIN_MODE", "full")              # "quick" or "full"
REFRESH_DATA = os.environ.get("REFRESH_DATA", "1") != "0"  # pull fresh nflverse data first
assert MODE in ("quick", "full"), "MODE must be 'quick' or 'full'"
FULL = MODE == "full"
print(f"MODE = {MODE}, REFRESH_DATA = {REFRESH_DATA}")

MODE = full, REFRESH_DATA = True


In [2]:
# ---- setup and helpers ------------------------------------------------------------------------------
import contextlib, importlib, json, re, shutil, subprocess, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 80)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nflelo").is_dir())
os.chdir(ROOT)
for p in (str(ROOT), str(ROOT / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)

from nflelo import config, data as elo_data                      # noqa: E402
from nflelo.elo import run_elo                                    # noqa: E402
from nflelo.ml import data as mldata, registry                    # noqa: E402
from nflelo.ml.eval import windows                                # noqa: E402

REAL_RUNS = registry.RUNS_DIR                                     # the committed registry: read-only here
CAPTURE = ROOT / "data" / "raw" / "ml" / "_train_all_runs"        # gitignored: where this notebook's run records go
shutil.rmtree(CAPTURE, ignore_errors=True)
CAPTURE.mkdir(parents=True)


def git_dirty() -> set:
    out = subprocess.run(["git", "status", "--porcelain", "--untracked-files=all"], cwd=ROOT,
                         capture_output=True, text=True, check=True).stdout
    return set(out.splitlines())


GIT_AT_START = git_dirty()
RUNS_AT_START = sorted(p.name for p in REAL_RUNS.glob("*.json"))

# Every script writes its scored runs through registry.log_run. Send them to CAPTURE instead of the registry,
# and refuse anything that touches the locked 2020-2025 seasons.
_real_log_run = registry.log_run


def _captured_log_run(*args, **kw):
    seasons = kw.get("seasons")
    if seasons is not None and windows.touches_holdout(seasons):
        raise RuntimeError(f"refusing to log a run on locked holdout seasons {seasons}")
    kw["runs_dir"] = CAPTURE
    return _real_log_run(*args, **kw)


registry.log_run = _captured_log_run

SECTION_SECONDS: dict = {}
ALL_RECS: list = []


@contextlib.contextmanager
def section(name: str):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        SECTION_SECONDS[name] = SECTION_SECONDS.get(name, 0.0) + time.perf_counter() - t0
        print(f"\n[{name}: {SECTION_SECONDS[name]:.0f}s]")


def run_main(module: str, argv=(), fn: str = "main", *args, **kw):
    # Run one script function (default: its main) with the given argv; returns the run records it produced.
    mod = importlib.import_module(module)
    before = {p.name for p in CAPTURE.glob("*.json")}
    old_argv, old_dir = sys.argv, registry.RUNS_DIR
    sys.argv, registry.RUNS_DIR = [f"{module}.py", *argv], CAPTURE   # so "runs written to ..." messages are true
    try:
        rc = getattr(mod, fn)(*args, **kw)
    except SystemExit as e:
        rc = e.code
    finally:
        sys.argv, registry.RUNS_DIR = old_argv, old_dir
    if rc not in (0, None, True):
        raise RuntimeError(f"{module}.{fn}{tuple(argv)} returned {rc!r}")
    recs = [json.loads(p.read_text()) for p in sorted(CAPTURE.glob("*.json")) if p.name not in before]
    ALL_RECS.extend(recs)
    return recs


# ---- "does it match the record?" ----------------------------------------------------------------------
SKIP_KEYS = re.compile(r"timing|seconds|_s$|elapsed|runtime", re.I)
PRIMARY = ["brier", "crps", "mse", "mae", "brier_playoffs", "ope_epa.diff", "logloss", "wmse", "loglik"]


def flat(d, prefix=""):
    out = {}
    for k, v in (d or {}).items():
        key = f"{prefix}{k}"
        if SKIP_KEYS.search(key):
            continue
        if isinstance(v, bool):
            continue
        if isinstance(v, (int, float)) and np.isfinite(v):
            out[key] = float(v)
        elif isinstance(v, dict):
            out.update(flat(v, key + "."))
    return out


def registry_ref(name: str, label: str):
    best = None
    for r in registry.load_runs(REAL_RUNS):
        if r["name"] == name and r["label"] == label and not r.get("holdout"):
            if best is None or r["timestamp"] >= best["timestamp"]:
                best = r
    return best


TOL_ABS, TOL_REL = 5e-5, 2e-3
QUICK_EXPECTED = {"m7a_audit"}    # depends on the bootstrap size, which quick mode shrinks     # nondeterminism allowance: |diff| <= max(5e-5, 0.2% of the record value)


def match_table(recs, title="Retrained vs the committed development record"):
    rows = []
    for r in recs:
        ref = registry_ref(r["name"], r["label"])
        if ref is None:
            rows.append({"run": r["name"], "key metric": "", "record": np.nan, "retrained": np.nan, "diff": np.nan,
                         "largest diff": np.nan, "on metric": "", "same games": "", "status": "NO RECORD"})
            continue
        a, b = flat(r["metrics"]), flat(ref["metrics"])
        keys = [k for k in b if k in a and k != "n"]
        bad = [k for k in keys if abs(a[k] - b[k]) > max(TOL_ABS, TOL_REL * abs(b[k]))]
        key = next((k for k in PRIMARY if k in keys), keys[0] if keys else None)
        worst = max(keys, key=lambda k: abs(a[k] - b[k]) / max(TOL_ABS, TOL_REL * abs(b[k]))) if keys else None
        if worst is not None and abs(a[worst] - b[worst]) < 1e-12:
            worst = None                                      # exact match on every metric
        same_n = r["n"] == ref["n"]
        rows.append({"run": r["name"], "key metric": key or "", "record": b.get(key, np.nan),
                     "retrained": a.get(key, np.nan), "diff": a[key] - b[key] if key else np.nan,
                     "largest diff": a[worst] - b[worst] if worst else 0.0, "on metric": worst or "(none: exact)",
                     "same games": "yes" if r["games_hash"] == ref["games_hash"] else ("n only" if same_n else "NO"),
                     "status": "OK" if not bad and same_n else
                               ("differs (quick bootstrap)" if not FULL and r["name"] in QUICK_EXPECTED else "DIFFERS")})
    t = pd.DataFrame(rows)
    n_ok = int(t["status"].isin(["OK"]).sum())
    n_exp = int(t["status"].str.startswith("differs (quick").sum())
    display(Markdown(f"**{title}**: {n_ok} of {len(t)} runs match "
                     f"(tolerance max({TOL_ABS:g}, {TOL_REL:.1%} of the record) on every metric)"
                     + (f"; {n_exp} expected to differ in quick mode." if n_exp else ".")))
    display(t.style.format({c: "{:.5f}" for c in ("record", "retrained", "diff", "largest diff")}, na_rep="").hide(axis="index"))
    return t


# ---- locked holdout results, loaded not recomputed --------------------------------------------------
def _fmt_metrics(r):
    bits = []
    for k, v in (r.get("metrics") or {}).items():
        if k == "n" or SKIP_KEYS.search(k):
            continue
        if isinstance(v, bool):
            continue
        if isinstance(v, (int, float)):
            bits.append(f"{k} {v:.4f}")
        elif isinstance(v, dict) and "diff" in v and "ci_low" in v:
            bits.append(f"{k} {v['diff']:+.4f} [{v['ci_low']:+.4f}, {v['ci_high']:+.4f}]")
        elif isinstance(v, dict) and "brier" in v:
            bits.append(f"{k}: brier {v['brier']:.4f}" + (f", ECE {v['cal']['ece']:.4f}" if "cal" in v else ""))
    for k, v in (r.get("comparisons") or {}).items():
        if isinstance(v, dict) and "diff" in v and "ci_low" in v:
            bits.append(f"{k} {v['diff']:+.4f} [{v['ci_low']:+.4f}, {v['ci_high']:+.4f}]")
    return "; ".join(bits[:9]) or "(descriptive tables only; see the record)"


def locked_holdout(prefixes, only=None, skip=(), note=""):
    # Show the committed 2020-2025 holdout records whose names start with a prefix (latest per name).
    latest = {}
    for r in registry.load_runs(REAL_RUNS):
        if r.get("holdout") and r["name"].startswith(tuple(prefixes)) and (only is None or r["name"].endswith(tuple(only))) and not r["name"].endswith(tuple(skip) or ("\0",)):
            if r["name"] not in latest or r["timestamp"] >= latest[r["name"]]["timestamp"]:
                latest[r["name"]] = r
    rows = []
    for name, r in sorted(latest.items()):
        parts = name.split("_")
        shown = "_".join([parts[0]] + parts[2:]) if r["label"] == "holdout" and len(parts) > 2 else name
        lo, hi = r["window"]["seasons"]
        rows.append({"record": shown, "seasons": f"{lo}-{hi}", "n": r["n"], "result (diff = [95% CI])": _fmt_metrics(r),
                     "committed": r["timestamp"][:10]})
    display(Markdown("**Locked 2020-2025 holdout, loaded from `experiments/runs/` (not recomputed)**" + (f". {note}" if note else "")))
    display(pd.DataFrame(rows).style.hide(axis="index"))


def runtime_note():
    return "  ·  ".join(f"{k}: {v:.0f}s" for k, v in SECTION_SECONDS.items())


print(f"repo root: {ROOT}")
print(f"registry: {len(RUNS_AT_START)} committed run files (read-only); this notebook's runs go to {CAPTURE.relative_to(ROOT)}/")
print(f"git changes before the run: {sorted(GIT_AT_START) or 'none'}")

repo root: /Users/walker_tracy/Desktop/Old Projects/NFLELO
registry: 167 committed run files (read-only); this notebook's runs go to data/raw/ml/_train_all_runs/
git changes before the run: ['?? notebooks/train_all_models.ipynb', '?? scripts/train_all.sh']


## 1. Data refresh

Pulls the newest nflverse schedule (and, in `full`, the current and previous season of every cached dataset),
then rebuilds `games` and the Elo ratings **in memory**. `data/games.csv` and `outputs/` are not touched: the
rebuilt games go to `data/raw/train_all/` (gitignored) and the scripts below are pointed at them.

In [3]:
CURRENT = 2026                      # the season the production models predict (scripts/ml_predict.SEASON)

with section("1. Data refresh"):
    if REFRESH_DATA:
        plan = {s: (list(mldata.FETCHERS) if FULL else ["pbp", "schedules"]) for s in ([CURRENT - 1, CURRENT] if FULL else [CURRENT])}
        rows = []
        for season, datasets in plan.items():
            for ds in datasets:
                if ds == "players" or season < mldata.FIRST_SEASON.get(ds, 0):
                    continue
                try:
                    r = mldata.refresh_one(ds, season)
                    rows.append({"dataset": ds, "season": season, "rows": r["rows"], "previous rows": r["old_rows"],
                                 "changed": r["changed"]})
                except Exception as e:                                   # e.g. a dataset not published yet
                    rows.append({"dataset": ds, "season": season, "rows": None, "previous rows": None,
                                 "changed": f"skipped: {type(e).__name__}"})
        if FULL:
            r = mldata.refresh_one("players", mldata.SNAPSHOT_SEASON)
            rows.append({"dataset": "players", "season": "snapshot", "rows": r["rows"], "previous rows": r["old_rows"],
                         "changed": r["changed"]})
        display(pd.DataFrame(rows))
    else:
        print("REFRESH_DATA is off: using the cached nflverse files.")

    sched_raw = elo_data.fetch_nflverse_schedules(refresh=REFRESH_DATA)
    nflverse = elo_data.nflverse_to_games(sched_raw)
    elo538 = elo_data.load_538_games(seasons=(1970, 2022))
    games_fresh = elo_data.build_games(elo538, nflverse)
    FRESH_DIR = ROOT / "data" / "raw" / "train_all"
    FRESH_DIR.mkdir(parents=True, exist_ok=True)
    FRESH_CSV = FRESH_DIR / "games_fresh.csv"
    games_fresh.to_csv(FRESH_CSV, index=False)
    # From here on every script that calls nflelo.data.load_games() reads the freshly built games.
    elo_data.load_games = lambda csv=FRESH_CSV: pd.read_csv(csv, parse_dates=["date"])

    committed = pd.read_csv(config.GAMES_CSV, parse_dates=["date"])
    new_ids = set(games_fresh["game_id"]) - set(committed["game_id"])
    both = games_fresh[games_fresh["game_id"].isin(committed["game_id"])].merge(
        committed, on="game_id", suffixes=("", "_old"))
    changed = both[(both["home_score"] != both["home_score_old"]) | (both["away_score"] != both["away_score_old"])]
    print(f"games rebuilt in memory: {len(games_fresh):,} games, seasons {games_fresh.season.min()}-{games_fresh.season.max()}")
    print(f"vs committed data/games.csv: {len(new_ids)} new finished games, {len(changed)} games with a changed score")

    ELO, ELO_STATE = run_elo(games_fresh, config.DEFAULT_CONFIG)
    last = ELO.sort_values(["date", "game_id"]).iloc[-1]
    print(f"Elo v2 rebuilt through {last['date']:%Y-%m-%d} (season {last['season']} week {last['week']}); "
          f"current home-field advantage {ELO_STATE['hfa']:.1f} Elo points")
    long = pd.concat([ELO[["date", "game_id", "home", "home_post"]].set_axis(["date", "game_id", "team", "post"], axis=1),
                      ELO[["date", "game_id", "away", "away_post"]].set_axis(["date", "game_id", "team", "post"], axis=1)])
    now_ratings = long.sort_values(["date", "game_id"], kind="stable").groupby("team")["post"].last().sort_values(ascending=False)
    display(now_ratings.head(10).round(0).astype(int).rename("Elo").to_frame().T)

,dataset,season,rows,previous rows,changed
0,pbp,2025,48771.0,48771.0,False
1,schedules,2025,285.0,285.0,False
2,player_stats,2025,19422.0,19422.0,False
3,depth_charts,2025,554215.0,554215.0,False
4,injuries,2025,6068.0,6068.0,False
5,rosters_weekly,2025,46849.0,46849.0,False
6,participation,2025,45184.0,45184.0,False
7,pbp,2026,16212.0,16212.0,False
8,schedules,2026,272.0,272.0,False
9,player_stats,2026,6417.0,6417.0,False


games rebuilt in memory: 13,777 games, seasons 1970-2026
vs committed data/games.csv: 14 new finished games, 0 games with a changed score
Elo v2 rebuilt through 2026-10-18 (season 2026 week 6); current home-field advantage 35.0 Elo points


team,SF,BUF,MIN,SEA,JAX,DEN,NE,KC,CHI,PIT
Elo,1643,1633,1624,1609,1603,1602,1594,1577,1554,1550



[1. Data refresh: 4s]


## 2. Elo v2 and the baselines (coin flip, home-team rate, betting market)

Elo is the project's starting model: K 20, season regression 0.40, a home-field advantage that is learned online.
The betting market (vig removed) is only a benchmark, never a feature. Elo itself has no training step beyond the
rebuild above, so this section scores it, with the three baselines, on the development seasons 2006-2019. Result
on record: Elo scores 0.2178 Brier vs the market's 0.2106.

In [4]:
import ml_baselines

with section("2. Elo v2 + baselines"):
    # DEV 2006-2019 only. (The script's own reproduction step re-scores 2010-2025, so it is not called.)
    recs_baselines = run_main("ml_baselines", [], "dev", elo_data.load_games(), True)
match_table(recs_baselines)
locked_holdout(["m3_"], only=("elo_v2", "market"),
               note="Elo and the market on the 1,615 holdout games that have moneylines.")


== (b) DEV, REG 2006-2019: identical games (those with moneylines) ==
3450 of 3584 REG games have moneylines; all four baselines are scored on those 3450.
model            n    brier  logloss     acc     ece
market        3450   0.2106   0.6096   0.663  0.0182
elo_v2        3450   0.2178   0.6255   0.641  0.0154
home_rate     3450   0.2451   0.6847   0.566  0.0069
coin_flip     3450   0.2493   0.6931   0.434  0.0659

Elo v2 minus market, Brier: +0.0072, 95% paired bootstrap CI [+0.0044, +0.0101] (2000 reps, seed 20261003); excludes zero.
Note: Elo v2 was tuned on 1980-2009, so 2006-2009 are in-sample for Elo.

[2. Elo v2 + baselines: 0s]


**Retrained vs the committed development record**: 4 of 4 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
coin_flip,brier,0.24935,0.24935,0.00000,0.00000,(none: exact),yes,OK
elo_v2,brier,0.21782,0.21782,0.00000,0.00000,(none: exact),yes,OK
home_rate,brier,0.24514,0.24514,0.00000,0.00000,(none: exact),yes,OK
market,brier,0.21060,0.21060,0.00000,0.00000,(none: exact),yes,OK


**Locked 2020-2025 holdout, loaded from `experiments/runs/` (not recomputed)**. Elo and the market on the 1,615 holdout games that have moneylines.

record,seasons,n,result (diff = [95% CI]),committed
m3_elo_v2,2020-2025,1615,brier 0.2231; logloss 0.6392; accuracy 0.6373; ece 0.0402,2026-10-03
m3_market,2020-2025,1615,brier 0.2096; logloss 0.6081; accuracy 0.6689; ece 0.0250,2026-10-03


## 3. M3: the game model (A4s) and the no-Elo study

Win probability from a logistic regression on Elo, opponent-adjusted EPA and a starting-quarterback adjustment.
The script runs the whole experiment ladder A0-A6 and picks A4s by the one-standard-error rule. **Passed** on the
locked seasons (Brier 0.2195 vs Elo 0.2231). The no-Elo study asks what the model would score without Elo; it is
descriptive and was never a pass/fail test.

In [5]:
with section("3a. M3 game model ladder"):
    recs_m3 = []
    if FULL:
        recs_m3 += run_main("ml_tune_ratings")           # re-tunes the EPA-rating and QB knobs on 2000-2005
    recs_m3 += run_main("ml_m3")
match_table(recs_m3)

== Stage 1: team ratings, next-week EPA margin, REG 2000-2005 ==
1518 games, 102 weekly fits per setting, grid 8 lambda x 10 half-life x 5 rho = 400 settings
Predict-zero MSE: 0.14822
 lam  half_life     rho    n     mse  mse_zero      r2    corr
 100         48 0.25000 1518 0.13043   0.14822 0.12002 0.35075
 100         24 0.25000 1518 0.13046   0.14822 0.11979 0.35026
 100         96 0.25000 1518 0.13046   0.14822 0.11976 0.35048
 100         12 0.50000 1518 0.13051   0.14822 0.11947 0.34905
 100       1000 0.25000 1518 0.13054   0.14822 0.11927 0.34987
 100         16 0.50000 1518 0.13054   0.14822 0.11923 0.34883
 100         16 0.25000 1518 0.13061   0.14822 0.11879 0.34873
 100         24 0.50000 1518 0.13074   0.14822 0.11788 0.34704
  50         48 0.25000 1518 0.13076   0.14822 0.11779 0.34964
  50         96 0.25000 1518 0.13077   0.14822 0.11767 0.34936
Chosen: lambda 100, half_life 48 weeks, rho 0.25
  best MSE by lam: 25: 0.13121, 50: 0.13076, 100: 0.13043, 200: 0.13083, 4


== Stage 2: QB shrinkage k (half_life 48, rho 0.25) ==
3036 starter-games, 87125 dropbacks. Replacement priors (EPA/dropback): 2000: -0.053, 2001: -0.128, 2002: -0.091, 2003: -0.066, 2004: -0.089, 2005: -0.118
   k  n_games  dropbacks    wmse
  25     3036      87125 0.11073
  50     3036      87125 0.10955
 100     3036      87125 0.10918
 200     3036      87125 0.11027
 300     3036      87125 0.11170
 400     3036      87125 0.11301
 600     3036      87125 0.11512
 800     3036      87125 0.11668
1200     3036      87125 0.11878
1600     3036      87125 0.12013
3200     3036      87125 0.12265
Chosen: k = 100 pseudo-dropbacks

Timing: load 0.4s, stage 1 23.4s (400 settings), stage 2 0.3s, total 24.1s
Code constants (opponent_adjust.TUNED, qb.TUNED_K): match
Run written to data/raw/ml/_train_all_runs/20261020T031409Z_m3_tune_ratings.json


DEV 2006-2019: 3450 REG games with moneylines (training: REG 2001..S-1, 1272 games before 2006).
Elo v2 on these games: Brier 0.2178. Feature build 2.5s (elo 0.0s, raw_epa 0.2s, ratings 0.2s, qb 2.1s)

A0 check: logistic on elo_logit 0.21783 vs Elo 0.21782 (diff +0.00001; tolerance 0.001)



Ladder, DEV 2006-2019 (n = 3450). diff = model minus Elo v2 Brier, 95% paired bootstrap CI (2,000 reps, seed 20261003); 1SE = within one SE of the best.
step model                                            brier  logloss    acc     ece  diff vs Elo [95% CI]          1SE
     Elo v2 (reference)                              0.2178   0.6255  0.641  0.0154
A0   elo_logit                                       0.2178   0.6257  0.645  0.0175  +0.0000 [-0.0003, +0.0003]      
A1   elo_logit + raw_epa_margin                      0.2180   0.6260  0.645  0.0165  +0.0001 [-0.0002, +0.0005]      
A2   elo_logit + adj_epa_margin                      0.2177   0.6254  0.644  0.0124  -0.0002 [-0.0008, +0.0005]      
A3   elo_logit + adj_pass_margin + adj_rush_margin   0.2180   0.6262  0.639  0.0157  +0.0002 [-0.0006, +0.0010]      
A4   elo_logit + adj_epa_margin + qb_home + qb_awa   0.2150   0.6195  0.647  0.0125  -0.0028 [-0.0045, -0.0011]   yes
A4s  elo_logit + adj_epa_margin + qb_diff            


Timing: load 1.1s, features 2.5s, ladder fits 7.4s, total 11.1s. One weekly update (2019 week 10, 13 games: Elo, ratings, QB values, predict): 0.50s, plus 0.00s for the once-a-season refit. Features match the ladder: True.



10 runs written to data/raw/ml/_train_all_runs/

[3a. M3 game model ladder: 37s]


**Retrained vs the committed development record**: 11 of 11 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
m3_tune_ratings,mse,0.13043,0.13043,0.00000,0.00000,(none: exact),yes,OK
m3_A0,brier,0.21783,0.21783,0.00000,0.00000,(none: exact),yes,OK
m3_A1,brier,0.21796,0.21796,0.00000,0.00000,(none: exact),yes,OK
m3_A2,brier,0.21765,0.21765,0.00000,0.00000,(none: exact),yes,OK
m3_A3,brier,0.21802,0.21802,0.00000,0.00000,(none: exact),yes,OK
m3_A4,brier,0.21500,0.21500,0.00000,0.00000,(none: exact),yes,OK
m3_A4b,brier,0.21715,0.21715,0.00000,0.00000,(none: exact),yes,OK
m3_A4bs,brier,0.21717,0.21717,0.00000,0.00000,(none: exact),yes,OK
m3_A4s,brier,0.21509,0.21509,0.00000,0.00000,(none: exact),yes,OK
m3_A5,brier,0.21498,0.21498,0.00000,0.00000,(none: exact),yes,OK


,run,key metric,record,retrained,diff,largest diff,on metric,same games,status
0,m3_tune_ratings,mse,0.130425,0.130425,0.0,0.0,(none: exact),yes,OK
1,m3_A0,brier,0.217827,0.217827,0.0,0.0,(none: exact),yes,OK
2,m3_A1,brier,0.217956,0.217956,0.0,0.0,(none: exact),yes,OK
3,m3_A2,brier,0.217653,0.217653,0.0,0.0,(none: exact),yes,OK
4,m3_A3,brier,0.218017,0.218017,0.0,0.0,(none: exact),yes,OK
5,m3_A4,brier,0.214999,0.214999,0.0,0.0,(none: exact),yes,OK
6,m3_A4b,brier,0.217145,0.217145,0.0,0.0,(none: exact),yes,OK
7,m3_A4bs,brier,0.217175,0.217175,0.0,0.0,(none: exact),yes,OK
8,m3_A4s,brier,0.215086,0.215086,0.0,0.0,(none: exact),yes,OK
9,m3_A5,brier,0.214984,0.214984,0.0,0.0,(none: exact),yes,OK


In [6]:
with section("3b. M3 no-Elo study"):
    recs_noelo = run_main("ml_m3_noelo")
match_table(recs_noelo)
locked_holdout(["m3_"], only=("A4s", "A4bs"), note="A4s is the live headline model; A4bs is its Wednesday-only twin.")

DEV 2006-2019: n = 3450, games hash 52b8194df3a2ebf4. Elo v2 Brier 0.2178; A4s recomputed 0.2151.



id  model                                                        brier  logloss    acc    ece    p95  vs Elo [95% CI]             vs A4s [95% CI]             
N1  adj_epa_margin                                              0.2209   0.6329  0.635 0.0183 0.0241  +0.0030 [+0.0007, +0.0053]  +0.0058 [+0.0036, +0.0080]
N2  adj_epa_margin + qb_delta_diff                              0.2183   0.6275  0.643 0.0163 0.0250  +0.0005 [-0.0021, +0.0033]  +0.0033 [+0.0016, +0.0049]
N3  adj_epa_margin + qb_delta_diff + rest_diff + neutral        0.2182   0.6271  0.647 0.0120 0.0250  +0.0004 [-0.0024, +0.0033]  +0.0031 [+0.0013, +0.0049]
N4  adj_pass_margin + adj_rush_margin + qb_delta_diff + rest_   0.2188   0.6284  0.643 0.0104 0.0243  +0.0010 [-0.0018, +0.0039]  +0.0037 [+0.0019, +0.0056]
N5  GBM: adj_pass_margin + adj_rush_margin + qb_delta_diff +    0.2209   0.6339  0.640 0.0208 0.0260  +0.0030 [-0.0003, +0.0066]  +0.0058 [+0.0032, +0.0085]
N6  adj_epa_margin + qb_delta_diff + raw_epa_margin    


Best no-Elo model: N3 (Brier 0.2182).
(a) Brier by week bucket:
  weeks       n       N3       N2      A4s      Elo  N3 - Elo [95% CI]             A4s - N2 [95% CI]           
  1-4       751   0.2277   0.2281   0.2243   0.2229  +0.0049 [-0.0005, +0.0100]    -0.0038 [-0.0073, -0.0004]
  5-9       969   0.2167   0.2173   0.2126   0.2140  +0.0027 [-0.0022, +0.0080]    -0.0047 [-0.0081, -0.0016]
  10-18    1730   0.2149   0.2147   0.2125   0.2178  -0.0028 [-0.0066, +0.0011]    -0.0022 [-0.0045, +0.0004]
  all    A4s - N2: -0.0033 [-0.0049, -0.0016]
(b) corr(logit p_N3, elo_logit) = 0.861; by bucket 1-4 0.850, 5-9 0.862, 10-18 0.869; corr with A4s logit 0.943; adj_epa_margin vs elo_logit 0.902
(c) A4s refit on one week bucket's training rows (2019 fit ± bootstrap SE; median over the 14 refits):
  weeks 1-4    n_train  1107  intercept +0.023 ±0.108 (+0.199)  elo_logit +1.041 ±0.252 (+0.474)  adj_epa_margin -0.361 ±1.182 (+2.502)  qb_delta_diff +2.581 ±1.276 (+3.347)
  weeks 5-9    n_train 


6 runs written to data/raw/ml/_train_all_runs/

[3b. M3 no-Elo study: 12s]


**Retrained vs the committed development record**: 6 of 6 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
m3_noelo_N1,brier,0.22086,0.22086,0.00000,0.00000,(none: exact),yes,OK
m3_noelo_N2,brier,0.21835,0.21835,0.00000,0.00000,(none: exact),yes,OK
m3_noelo_N3,brier,0.21820,0.21820,0.00000,0.00000,(none: exact),yes,OK
m3_noelo_N4,brier,0.21880,0.21880,0.00000,0.00000,(none: exact),yes,OK
m3_noelo_N5,brier,0.22086,0.22086,0.00000,0.00000,(none: exact),yes,OK
m3_noelo_N6,brier,0.21849,0.21849,0.00000,0.00000,(none: exact),yes,OK


**Locked 2020-2025 holdout, loaded from `experiments/runs/` (not recomputed)**. A4s is the live headline model; A4bs is its Wednesday-only twin.

record,seasons,n,result (diff = [95% CI]),committed
m3_A4bs,2020-2025,1615,brier 0.2216; logloss 0.6360; accuracy 0.6373; ece 0.0311,2026-10-03
m3_A4s,2020-2025,1615,"brier 0.2195; logloss 0.6310; accuracy 0.6429; ece 0.0283; A4s_minus_elo_brier -0.0035 [-0.0063, -0.0007]; A4s_minus_elo_logloss -0.0082 [-0.0147, -0.0016]; A4s_minus_market_brier +0.0099 [+0.0061, +0.0137]; A4s_minus_market_logloss +0.0229 [+0.0144, +0.0314]; A4s_minus_A4bs_brier -0.0020 [-0.0042, +0.0002]",2026-10-03


## 4. M3b: Kalman-filter team strength (C2d, the shadow model)

A state-space rating that carries uncertainty, added to Elo and the QB term (C2d). On the development seasons it
beat A4s by 0.00095 Brier, just short of the pre-set bar, so it was **never run on the holdout**. It runs as a
shadow model in the live 2026 ledger. The `full` mode re-tunes the Kalman knobs on 2000-2005 (about 4 minutes).

In [7]:
with section("4. M3b Kalman (C2d)"):
    recs_m3b = []
    if FULL:
        recs_m3b += run_main("ml_m3b", ["tune"])
    recs_m3b += run_main("ml_m3b", ["dev"])
match_table(recs_m3b)
print("No holdout result exists for M3b by design: it narrowly missed its development bar.")

== Kalman tuning, 2000-2005 (filter from 1999), 1586 games, 195.1s, 7983 evaluations ==
starts (joint loglik, margin loglik, knobs):
    -6072.17   -6368.85  q_week 0.8516, gamma 0.4948, q_season 11.73, sigma_m 12.36, sigma_e 13.49, rho 0.807, scale 39.84, q_hfa 0.0003818
    -6071.90   -6368.60  q_week 0.7137, gamma 0.4836, q_season 14.42, sigma_m 12.36, sigma_e 13.51, rho 0.8071, scale 39.88, q_hfa 0.0003564
    -6071.90   -6368.59  q_week 0.7215, gamma 0.4832, q_season 14.43, sigma_m 12.36, sigma_e 13.52, rho 0.8071, scale 39.92, q_hfa 0.0003456
    -6071.95   -6368.80  q_week 0.7285, gamma 0.4842, q_season 14.11, sigma_m 12.36, sigma_e 13.35, rho 0.8066, scale 39.49, q_hfa 0.0003647
margin-only optima from the same starts (not used; flat ridge):
    -6371.25  q_week 0.899, gamma 0.5332, q_season 5.444, sigma_m 12.55, sigma_e 15.87, rho 0.6572, scale 58.77, q_hfa 0.01058
    -6372.55  q_week 0.1288, gamma 0.5534, q_season 1.082, sigma_m 13.21, sigma_e 20.99, rho -0.8819, scale 29.21

== QB extras tuning ==
baseline (M3 QB value, k 100) on 2000-2005: wMSE 0.10918 (3036 starter-games); M3 code path 0.10918


(b) experience offsets (-0.0541, 0.0159, 0.1412, 0.0648) (buckets (0, 100, 500, 2000), starter-games [210, 409, 506, 1911]): wMSE 0.10839 vs 0.10918


(b2) experience offsets tuned on the 351 changed-starter games: (-0.0651, 0.08, 0.0329, 0.0318): wMSE 0.11445 vs 0.11588 without (the all-starter offsets give 0.11505 here)


(c) aging (peak, slope before, slope after) = (31.0, 0.14, 0.0): wMSE 0.10810 vs 0.10918; 4361 of 4361 QB-games have a birth date; top 5:
 peak  before  after     wmse
   31    0.14  0.000 0.108098
   31    0.12  0.000 0.108109
   31    0.14 -0.005 0.108109
   31    0.12 -0.005 0.108120
   31    0.14 -0.010 0.108121


...


Wednesday twin C2d_last: Brier 0.2161; minus A4bs -0.0011 [-0.0021, -0.0001]; minus A4s +0.0010 [-0.0005, +0.0026]

Coefficients (last DEV refit; range over the 14 refits):
  A4s (2019 fit, n 4600): intercept +0.048 [+0.021, +0.091], elo_logit +0.842 [+0.619, +0.871], adj_epa_margin +1.087 [+0.872, +2.214], qb_delta_diff +3.607 [+2.740, +3.759]
  C1 (2019 fit, n 4600): intercept +0.047 [+0.018, +0.091], elo_logit +0.840 [+0.613, +0.867], adj_epa_margin +1.055 [+0.842, +2.256], qb_delta_diff +3.616 [+2.740, +3.763], elo_logit_x_early1 +0.088 [-0.393, +0.294], adj_epa_margin_x_early1 +0.475 [-1.536, +1.115]
  C2a (2019 fit, n 4600): intercept -0.033 [-0.063, +0.017], kf_margin +0.151 [+0.110, +0.151], adj_epa_margin -0.622 [-0.632, +0.742], qb_delta_diff +3.359 [+2.559, +3.541]
  C2b (2019 fit, n 4600): intercept -0.064 [-0.086, -0.009], elo_logit +0.452 [+0.194, +0.511], kf_margin +0.101 [+0.065, +0.129], adj_epa_margin -0.899 [-0.899, +0.521], qb_delta_diff +3.481 [+2.616, +3.647]
  C


16 runs written to data/raw/ml/_train_all_runs/

[4. M3b Kalman (C2d): 227s]


**Retrained vs the committed development record**: 18 of 18 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
m3b_tune_kalman,loglik_per_game,-4.01550,-4.01550,-0.00000,0.00001,rmse,yes,OK
m3b_tune_qb,,,,,0.00000,(none: exact),yes,OK
m3b_A4bs,brier,0.21717,0.21717,0.00000,0.00000,(none: exact),yes,OK
m3b_A4s,brier,0.21509,0.21509,0.00000,0.00000,(none: exact),yes,OK
m3b_C1,brier,0.21517,0.21517,0.00000,0.00000,(none: exact),yes,OK
m3b_C2a,brier,0.21459,0.21459,-0.00000,-0.00000,ece_null_p95,yes,OK
m3b_C2b,brier,0.21415,0.21415,0.00000,0.00000,ece,yes,OK
m3b_C2c,brier,0.21452,0.21452,0.00000,0.00000,ece_null_p95,yes,OK
m3b_C2d,brier,0.21413,0.21413,0.00000,0.00000,(none: exact),yes,OK
m3b_C2k,brier,0.21667,0.21667,0.00000,0.00000,(none: exact),yes,OK


No holdout result exists for M3b by design: it narrowly missed its development bar.


## 5. M4: margin model and playoff odds

The margin model predicts the whole distribution of a game's final margin, aware of key numbers like 3 and 7. It
**failed** its holdout primary (the improvement over Elo's spread was within noise) but powers spreads and the
simulation. The playoff odds (20,000 simulated seasons with real NFL tiebreakers) **passed**. `quick` retrains the
margin model only; `full` also re-tunes the rest-week uncertainty and backtests the simulation on 2006-2019
(about 18 minutes).

In [8]:
with section("5. M4 margin + playoff odds"):
    recs_m4 = run_main("ml_m4", ["margin"])
    if FULL:
        recs_m4 += run_main("ml_m4", ["tune"])
        recs_m4 += run_main("ml_m4", ["backtest"])
match_table(recs_m4)
locked_holdout(["m4_"])

== Margin model, DEV 2006-2019, REG games with moneylines: n = 3450 (games hash 52b8194df3a2ebf4) ==
MAE  model 10.667   Elo spread (elo_diff/25) 10.731   market spread 10.487
     model minus Elo  -0.064 [-0.118, -0.009]  excludes zero
     model minus market +0.180 [+0.096, +0.268]
CRPS normal_continuous 7.6638   normal 7.6620   keynum 7.6480
     shape rule: best keynum; other minus best +0.0141 (SE 0.0042) -> chosen: keynum
shape      brier  logloss    acc    ece  ECE null 5-95%  minus A4s Brier [95% CI]       1SE
normal    0.2151   0.6199  0.648 0.0129  [0.010, 0.026]  +0.00004 [-0.00039, +0.00051]   yes
keynum    0.2153   0.6203  0.648 0.0141  [0.009, 0.025]  +0.00018 [-0.00036, +0.00074]   yes
A4s       0.2151   0.6197  0.647 0.0114  [0.010, 0.025]
Home team beats our spread, by predicted spread (5-95% range of a fair coin at that n):
  (-inf, -7]   n   213  0.545  [0.441, 0.559]  inside
  (-7, -3]     n   416  0.524  [0.459, 0.541]  inside
  (-3, 0]      n   573  0.475  [0.466,

  tune 2006 week 4: 210 games left, state 0.3s, 17 x 5,000 sims 15.6s


  tune 2006 week 8: 156 games left, state 0.3s, 17 x 5,000 sims 15.2s


  tune 2006 week 12: 96 games left, state 0.3s, 17 x 5,000 sims 15.4s


  tune 2006 week 16: 32 games left, state 0.3s, 17 x 5,000 sims 14.9s


...

== Playoff odds, DEV (in-sample for tau, descriptive) 2006-2019: 56 starts, 1792 team-starts, tau_rest 4.5 vs 0 ==
outcome    Brier tau  Brier 0  diff [95% CI, team-starts]    [95% CI, by season]        ECE  null 5-95%    
playoffs      0.1195   0.1213  -0.0017 [-0.0031, -0.0004]     [-0.0044, +0.0008]    0.018  [0.012, 0.028]
division      0.0882   0.0880  +0.0001 [-0.0011, +0.0013]     [-0.0017, +0.0022]    0.021  [0.011, 0.025]
seed1         0.0320   0.0317  +0.0003 [-0.0005, +0.0010]     [-0.0008, +0.0014]    0.012  [0.006, 0.015]
reach_sb      0.0465   0.0458  +0.0007 [-0.0001, +0.0016]     [-0.0004, +0.0019]    0.014  [0.004, 0.015]
win_sb        0.0281   0.0283  -0.0002 [-0.0006, +0.0002]     [-0.0008, +0.0003]    0.002  [0.002, 0.010]  (below p05)
made playoffs by start week (Brier tau / tau 0, ECE): week 4: 0.1732 / 0.1764, 0.043; week 8: 0.1383 / 0.1398, 0.032; week 12: 0.1041 / 0.1056, 0.031; week 16: 0.0625 / 0.0632, 0.026
made playoffs by season (Brier tau / tau 0): 2006: 

**Retrained vs the committed development record**: 5 of 5 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
m4_margin_keynum,brier,0.21526,0.21526,0.00000,0.00000,(none: exact),yes,OK
m4_margin_normal,brier,0.21513,0.21513,0.00000,0.00000,(none: exact),yes,OK
m4_tau_rest_tune_dev,brier_playoffs,0.11941,0.11941,0.00000,0.00000,(none: exact),yes,OK
m4_playoff_odds,brier,0.11954,0.11954,0.00000,0.00000,(none: exact),yes,OK
m4_playoff_odds_tau0,brier,0.12125,0.12125,0.00000,0.00000,(none: exact),yes,OK


**Locked 2020-2025 holdout, loaded from `experiments/runs/` (not recomputed)**

record,seasons,n,result (diff = [95% CI]),committed
m4_margin,2020-2025,1615,brier 0.2195; logloss 0.6309; accuracy 0.6410; ece 0.0348; crps 7.2541; mae 10.1261; mae_elo 10.1850; mae_market 9.7644,2026-10-05
m4_playoff_odds,2020-2025,768,brier 0.1234; brier_tau0 0.1258; ece 0.0282; tau_rest 4.5000,2026-10-05


## 6. M5: box-score player values and pregame lineups

Per-player values from game-level stats, summed over the expected lineup, tested as extra inputs to A4s on
2012-2019. A **negative result**: no step beats A4s (every confidence interval includes zero), so the player
features do not enter the game model. No holdout was run (nothing to test). `full` also re-tunes the value knobs
on 2001-2008.

In [9]:
with section("6. M5 box-score player values"):
    recs_m5 = []
    if FULL:
        recs_m5 += run_main("ml_tune_players")
    recs_m5 += run_main("ml_m5")
match_table(recs_m5)
print("No holdout result exists for M5 by design: nothing beat A4s on the development window.")

Inputs 1999-2008 loaded and credited in 5.2s: 125,134 player-games.
Era gaps (component recorded = 1):
season   1999  2000  2001  2002  2003  2004  2005  2006  2007  2008
db          1     1     1     1     1     1     1     1     1     1
rush        1     1     1     1     1     1     1     1     1     1
rb_recv     1     1     1     1     0     0     0     0     0     0
recv        1     1     1     1     0     0     0     0     0     0
sack        1     1     1     1     1     1     1     1     1     1
qbhit       0     0     0     0     0     0     0     1     1     1
tfl         1     1     1     1     0     0     0     0     0     1
ff          1     1     1     1     1     1     1     1     1     1
int         1     1     1     1     1     1     1     1     1     1
pd          1     1     1     1     1     1     1     1     1     1
fg          1     1     1     1     1     1     1     1     1     1



Box values, 2001-2008 anchors (next 4 games, same season). Chosen half-life 32 weeks (sum of MSE / MSE_replacement over groups, each at its best k):
  4: 8.4621, 8: 8.3128, 16: 8.2155, 32: 8.1871, 64: 8.1915, 128: 8.2020
group        n      k   skill       wmse  wmse_repl  k grid edge?
CB       11985     16   0.033   0.000875   0.000905  
DL       20691      8   0.092   0.000719   0.000791  
K         4079     16   0.134   0.000322   0.000371  
LB       15241      8   0.075   0.000590   0.000637  
QB        4468      4   0.297   0.015519   0.022091  
RB       11168     16   0.022   0.008117   0.008302  
S        10343     32   0.026   0.000548   0.000562  
TE        5698     16   0.048   0.000598   0.000629  
WR       12644     16   0.086   0.000674   0.000738  



With/without ridge, 2006-2008 anchors (next 4 team-games, residual EPA/play, plays-weighted MSE; lambda 1e7 = box values only):
            lam  mse_off  mse_def  mse_icpt_off  mse_icpt_def      mse  mse_icpt
     125.000000 0.053462 0.053461      0.060498      0.049883 0.053462  0.055189
     250.000000 0.051279 0.050907      0.059279      0.049896 0.051093  0.054587
     500.000000 0.049977 0.049404      0.058512      0.049892 0.049691  0.054201
    1000.000000 0.049251 0.048587      0.058030      0.049877 0.048919  0.053952
    2000.000000 0.048889 0.048237      0.057721      0.049858 0.048563  0.053789
    4000.000000 0.048760 0.048203      0.057518      0.049844 0.048481  0.053680
    8000.000000 0.048782 0.048332      0.057382      0.049833 0.048557  0.053607
   16000.000000 0.048877 0.048493      0.057294      0.049827 0.048685  0.053560
   64000.000000 0.049055 0.048693      0.057211      0.049822 0.048874  0.053516
10000000.000000 0.049155 0.048782      0.057177      0.049820

Logged data/raw/ml/_train_all_runs/20261020T033823Z_m5_tune_players.json


M5 window REG 2012-2019 with moneylines: n = 2047, games hash 890722aef13c4355.
Training: B0 on REG 2001..S-1 (as M3); every other step on REG 2009..S-1 (seasons 2009-2011 for the 2012 model up to 2009-2018 for 2019). Team-games with a zero lineup delta in 2009-2019: 0.
Status probabilities (estimated on 2009-2015 rank-1 starters):
status  doubtful  none    out  practice  probable  questionable
group                                                          
CB         0.019   1.0  0.006       1.0       1.0         0.679
DL         0.030   1.0  0.000       1.0       1.0         0.778
K          0.018   1.0  0.001       1.0       1.0         0.763
LB         0.013   1.0  0.000       1.0       1.0         0.764
OL         0.023   1.0  0.002       1.0       1.0         0.765
QB         0.040   1.0  0.012       1.0       1.0         0.601
RB         0.006   1.0  0.000       1.0       1.0         0.868
S          0.005   1.0  0.000       1.0       1.0         0.727
TE         0.052   1.0  0.

B0 vs the saved M3 ladder A4s predictions on these games: max |diff| 0.00e+00

Ladder (n = 2047; diff = step minus B0 Brier, 95% paired bootstrap CI, 2,000 reps, seed 20261003; ECE p95 = one-sided null at this n):
step train        brier  logloss    acc    ece    p95  ok  vs B0 [95% CI]                1SE
B0   2001..S-1   0.2147   0.6195  0.650 0.0152 0.0330   y  +0.0000 [+0.0000, +0.0000]   yes *
B0r  2009..S-1   0.2150   0.6202  0.650 0.0146 0.0329   y  +0.0004 [-0.0000, +0.0007]      
B1   2009..S-1   0.2149   0.6199  0.647 0.0185 0.0330   y  +0.0002 [-0.0006, +0.0011]   yes
B2   2009..S-1   0.2150   0.6202  0.650 0.0252 0.0334   y  +0.0004 [-0.0004, +0.0012]   yes
B3   2009..S-1   0.2154   0.6211  0.647 0.0239 0.0318   y  +0.0008 [-0.0004, +0.0019]      
B4   2009..S-1   0.2154   0.6211  0.649 0.0210 0.0329   y  +0.0008 [-0.0004, +0.0020]      
B5   2009..S-1   0.2155   0.6212  0.645 0.0263 0.0321   y  +0.0008 [-0.0003, +0.0020]      
B3f  2009..S-1   0.2155   0.6212  0.647 0.0227 


8 runs written to data/raw/ml/_train_all_runs/

[6. M5 box-score player values: 167s]


**Retrained vs the committed development record**: 9 of 9 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
m5_tune_players,half_life,32.00000,32.00000,0.00000,0.00000,(none: exact),yes,OK
m5_B0,brier,0.21466,0.21466,0.00000,0.00000,(none: exact),yes,OK
m5_B0r,brier,0.21502,0.21502,0.00000,0.00000,(none: exact),yes,OK
m5_B1,brier,0.21490,0.21490,0.00000,0.00000,(none: exact),yes,OK
m5_B2,brier,0.21505,0.21505,0.00000,0.00000,(none: exact),yes,OK
m5_B3,brier,0.21542,0.21542,0.00000,0.00000,(none: exact),yes,OK
m5_B3f,brier,0.21546,0.21546,0.00000,0.00000,(none: exact),yes,OK
m5_B4,brier,0.21541,0.21541,0.00000,0.00000,(none: exact),yes,OK
m5_B5,brier,0.21547,0.21547,0.00000,0.00000,(none: exact),yes,OK


No holdout result exists for M5 by design: nothing beat A4s on the development window.


## 7. M5b: on-field player ratings (RAPM)

Bayesian plus-minus ratings from all 22 players on every play (participation data, 2016+, CC BY-SA), tested
season-ahead against team ratings and box-score values. **Failed** on the holdout: it beats team ratings but loses
to box-score values summed over the on-field players, so it is kept as a descriptive player view only.

In [10]:
with section("7. M5b RAPM"):
    recs_m5b = run_main("ml_m5b", ["clean"])
    if FULL:
        recs_m5b += run_main("ml_m5b", ["tune"])
    recs_m5b += run_main("ml_m5b", ["dev"])
match_table(recs_m5b)
locked_holdout(["m5b_"])

Participation cleaning (kept = exactly 11 + 11 distinct GSIS IDs after the side repair):
 season era  clean_plays  joined  raw_11_11  kept  repaired  raw_drop_rate  drop_rate  games
   2016 ngs        28488   28488      24856 28367      3511         0.1275     0.0042    267
   2017 ngs        28086   28084      28051 28051         0         0.0012     0.0012    267
   2018 ngs        27629   27629      25442 27611      2169         0.0792     0.0007    267
   2019 ngs        27957   27957      26597 27932      1335         0.0486     0.0009    267
   2020 ngs        28744   28744      28715 28715         0         0.0010     0.0010    269
   2021 ngs        29557   29557      29535 29535         0         0.0007     0.0007    285
   2022 ngs        30439   30439      30412 30412         0         0.0009     0.0009    284
   2023 ftn        30065   30065      30040 30040         0         0.0008     0.0008    285
   2024 ftn        29513   29513      29489 29489         0         0.0008


Distinct players on the field per season:
 season era  players
   2016 ngs     1833
   2017 ngs     1851
   2018 ngs     1839
   2019 ngs     1858
   2020 ngs     1964
   2021 ngs     2034
   2022 ngs     1996
   2023 ftn     1905
   2024 ftn     1969
   2025 ftn     1936

Players per season and position group (group from that season's depth charts, then rosters):
group    CB   DL   K   LB   OL  QB   RB    S   TE   WR  unknown
season                                                         
2016    203  284  11  263  304  64  169  162  129  212       32
2017    195  301   5  269  305  65  163  164  128  218       38
2018    200  302   8  252  303  65  160  163  132  225       29
2019    211  296   8  264  296  66  157  171  130  236       23
2020    224  331   6  264  313  74  169  169  140  244       30
2021    230  338   7  291  335  72  167  167  143  260       24
2022    237  339   5  271  327  78  164  161  137  248       29
2023    226  320   5  269  313  76  148  159  138  232  

Start {'lam': {'CB': 4000.0, 'DL': 4000.0, 'LB': 4000.0, 'OL': 4000.0, 'QB': 4000.0, 'RB': 4000.0, 'S': 4000.0, 'TE': 4000.0, 'WR': 4000.0}, 'lam_other': 20000.0, 'decay': 0.5, 'situation': True, 'box_scale': 1.0}: MSE 1.940818


Sweep 1: MSE 1.938726  decay 1.0  box_scale 0.25  situation True  CB 1024000 DL 1024000 LB 2000 OL 2000 QB 250 RB 1000 S 4000 TE 4000 WR 8000


Sweep 2: MSE 1.938630  decay 1.0  box_scale 0.25  situation True  CB 32000 DL 1024000 LB 2000 OL 1000 QB 500 RB 1000 S 4000 TE 2000 WR 4000


...


Runs written to data/raw/ml/_train_all_runs/ (label m5bdev)

[7. M5b RAPM: 103s]


**Retrained vs the committed development record**: 6 of 6 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
m5b_tune,mse,1.93863,1.93863,0.00000,0.00000,(none: exact),yes,OK
m5b_dev_box,mse,1.94050,1.94050,0.00000,0.00000,(none: exact),yes,OK
m5b_dev_intercept,mse,1.94914,1.94914,0.00000,0.00000,(none: exact),yes,OK
m5b_dev_rapm,mse,1.93863,1.93863,0.00000,0.00000,(none: exact),yes,OK
m5b_dev_team,mse,1.94136,1.94136,0.00000,0.00000,(none: exact),yes,OK
m5b_dev_game_model,brier,0.21771,0.21771,0.00000,0.00000,(none: exact),yes,OK


**Locked 2020-2025 holdout, loaded from `experiments/runs/` (not recomputed)**

record,seasons,n,result (diff = [95% CI]),committed
m5b_box,2020-2025,177545,mse 1.8861; margin_mse 0.1439,2026-10-05
m5b_game_model,2020-2025,1615,brier 0.2193; brier_a4s 0.2195,2026-10-05
m5b_intercept,2020-2025,177545,mse 1.8958; margin_mse 0.1570,2026-10-05
m5b_rapm,2020-2025,177545,"mse 1.8871; margin_mse 0.1431; team -0.0018 [-0.0028, -0.0008]; box +0.0010 [+0.0002, +0.0018]; intercept -0.0087 [-0.0105, -0.0070]; team_vs_intercept -0.0070 [-0.0086, -0.0053]; box_vs_intercept -0.0097 [-0.0112, -0.0082]; box_vs_team -0.0028 [-0.0037, -0.0019]",2026-10-05
m5b_team,2020-2025,177545,mse 1.8888; margin_mse 0.1489,2026-10-05


## 8. M6: play outcome model

Predicts the whole yards-gained distribution of a play from pre-snap facts (down, distance, field position,
personnel, formation, team ratings). A gradient-boosted model and three small neural nets compete against a plain
situational baseline, scored by CRPS (lower is better). **Passed** on the holdout (CRPS about 0.018 better than the
baseline, in every season). Neural-net training is the slow part and is not bit-for-bit reproducible, so small
differences from the record are expected.

In [11]:
with section("8. M6 play model"):
    recs_m6 = run_main("ml_m6", ["fields"])
    recs_m6 += run_main("ml_m6", ["dev"])
match_table(recs_m6)
locked_holdout(["m6_"])

 season era  plays  raw_formations  form_missing  shotgun  pistol  under_center  other  box_missing  box_mean  box_sd  box_8plus  pers_missing    rb    te    wr    ol   p11    dl    lb    db  nickel_or_more
   2016 ngs  28367               7         0.009    0.592   0.040         0.355  0.005        0.001     6.371   1.059      0.124           0.0 1.100 1.237 2.606 5.055 0.587 3.201 3.009 4.786           0.688
   2017 ngs  28051               7         0.008    0.570   0.016         0.403  0.004        0.001     6.492   1.081      0.158           0.0 1.122 1.305 2.527 5.037 0.578 3.245 2.980 4.772           0.653
   2018 ngs  27611               7         0.009    0.607   0.020         0.362  0.002        0.001     6.374   1.050      0.121           0.0 1.108 1.260 2.584 5.031 0.621 3.412 2.706 4.880           0.740
   2019 ngs  27932               7         0.001    0.604   0.032         0.360  0.003        0.001     6.341   1.048      0.114           0.0 1.120 1.278 2.547 5.043 0.571

Inputs loaded in 26s: 111,961 plays 2016-2019


  situation 2018: 27,611 plays, 50s baseline 4.0540 gbm 4.0330 N0 4.0347 N1 4.0374 N2 4.0373


  situation 2019: 27,932 plays, 126s baseline 3.9953 gbm 3.9792 N0 3.9840 N1 3.9822 N2 3.9827


  call 2018: 27,611 plays, 70s baseline 3.9747 gbm 3.9574 N0 3.9572 N1 3.9578 N2 3.9562


...


Runs written to data/raw/ml/_train_all_runs/ (label m6dev)

[8. M6 play model: 408s]


**Retrained vs the committed development record**: 10 of 10 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
m6_call_N0,crps,3.93235,3.93235,0.00000,0.00000,(none: exact),yes,OK
m6_call_N1,crps,3.93095,3.93095,0.00000,0.00000,(none: exact),yes,OK
m6_call_N2,crps,3.93008,3.93008,0.00000,0.00000,(none: exact),yes,OK
m6_call_baseline,crps,3.94835,3.94835,0.00000,0.00000,(none: exact),yes,OK
m6_call_gbm,crps,3.93040,3.93040,0.00000,0.00000,(none: exact),yes,OK
m6_situation_N0,crps,4.00922,4.00922,0.00000,0.00000,(none: exact),yes,OK
m6_situation_N1,crps,4.00963,4.00963,0.00000,0.00000,(none: exact),yes,OK
m6_situation_N2,crps,4.00985,4.00985,0.00000,0.00000,(none: exact),yes,OK
m6_situation_baseline,crps,4.02446,4.02446,0.00000,0.00000,(none: exact),yes,OK
m6_situation_gbm,crps,4.00593,4.00593,0.00000,0.00000,(none: exact),yes,OK


**Locked 2020-2025 holdout, loaded from `experiments/runs/` (not recomputed)**

record,seasons,n,result (diff = [95% CI]),committed
m6_call_N0,2020-2025,177545,crps 3.7746; logloss 2.8336; tov_brier 0.0185; ece_first 0.0097,2026-10-05
m6_call_N1,2020-2025,177545,crps 3.7734; logloss 2.8315; tov_brier 0.0185; ece_first 0.0106,2026-10-05
m6_call_N2,2020-2025,177545,crps 3.7724; logloss 2.8313; tov_brier 0.0185; ece_first 0.0105,2026-10-05
m6_call_baseline,2020-2025,177545,crps 3.7902; logloss 2.8553; tov_brier 0.0186; ece_first 0.0076,2026-10-05
m6_call_gbm,2020-2025,177545,crps 3.7713; logloss 2.8300; tov_brier 0.0185; ece_first 0.0074,2026-10-05
m6_situation_N0,2020-2025,177545,crps 3.8461; logloss 2.9230; tov_brier 0.0186; ece_first 0.0094,2026-10-05
m6_situation_N1,2020-2025,177545,crps 3.8455; logloss 2.9214; tov_brier 0.0186; ece_first 0.0117,2026-10-05
m6_situation_N2,2020-2025,177545,crps 3.8448; logloss 2.9208; tov_brier 0.0186; ece_first 0.0109,2026-10-05
m6_situation_baseline,2020-2025,177545,crps 3.8617; logloss 2.9496; tov_brier 0.0186; ece_first 0.0078,2026-10-05
m6_situation_gbm,2020-2025,177545,crps 3.8436; logloss 2.9203; tov_brier 0.0186; ece_first 0.0068,2026-10-05


## 9. M7a: win probability and fourth-down decisions

Our own win-probability model (no betting inputs), plus field-goal, punt and fourth-down conversion models that
value going for it, kicking and punting. The win-probability model **passed** (Brier 0.1559 vs nflfastR's 0.1659).
The original conversion engine **failed** its calibration bar; the revised engine (v2) is evaluated on development
seasons below; its real test is the 2026 season, which cannot be read before 2027. This notebook only retrains and
scores on development seasons.

In [12]:
BOOT = 50 if FULL else 4               # bootstrap refits per season for the uncertainty bands (the record used 50)
with section("9. M7a win prob + fourth down"):
    recs_m7a = run_main("ml_m7a", ["dev", "--boot", str(BOOT)])
    recs_m7a += run_main("ml_m7a", ["v2-dev"])
match_table(recs_m7a, title=f"Retrained vs record ({BOOT} bootstrap refits per season; in quick mode the bands are rougher, the point estimates should not move)")
locked_holdout(["m7a_"], skip=("audit",), note="The conversion row is the original engine, which missed its calibration bar (ECE 0.027). v2 has no holdout result: its test is the live 2026 season.")

inputs through 2019: pbp 984,639 rows, A4s 3,584 games, WP states 529,839, M6 plays 111,961 (13s)


  season 2016 done (18s): wp 3s, kicking 15s


  season 2017 done (21s): wp 6s, kicking 14s


  bootstrap 2018 1/50 (17.7s)


  bootstrap 2018 2/50 (18.2s)


...


[9. M7a win prob + fourth down: 2209s]


**Retrained vs record (50 bootstrap refits per season; in quick mode the bands are rougher, the point estimates should not move)**: 14 of 14 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
m7a_audit,games,509.00000,509.00000,0.00000,0.00000,(none: exact),yes,OK
m7a_bench_nflfastr_wp,brier,0.15271,0.15271,0.00000,0.00000,(none: exact),yes,OK
m7a_bench_vegas_wp,brier,0.13798,0.13798,0.00000,0.00000,(none: exact),yes,OK
m7a_conversion,engine_on_attempts.n,1118.00000,1118.00000,0.00000,0.00000,(none: exact),yes,OK
m7a_fg,brier,0.12018,0.12018,0.00000,0.00000,(none: exact),yes,OK
m7a_punt,crps,5.71736,5.71736,0.00000,0.00000,(none: exact),yes,OK
m7a_wp,brier,0.14292,0.14292,0.00000,0.00000,(none: exact),yes,OK
m7a_wp_isotonic,brier,0.14403,0.14403,0.00000,0.00000,(none: exact),yes,OK
m7a_v2_ref_v1,brier,0.23050,0.23050,0.00000,0.00000,(none: exact),yes,OK
m7a_v2_v2a,brier,0.23030,0.23030,0.00000,0.00000,(none: exact),yes,OK


**Locked 2020-2025 holdout, loaded from `experiments/runs/` (not recomputed)**. The conversion row is the original engine, which missed its calibration bar (ECE 0.027). v2 has no holdout result: its test is the live 2026 season.

record,seasons,n,result (diff = [95% CI]),committed
m7a_bench_nflfastr_wp,2020-2025,233913,brier 0.1659; logloss 0.4898; ece 0.0119,2026-10-05
m7a_bench_vegas_wp,2020-2025,233913,brier 0.1497; logloss 0.4518; ece 0.0059,2026-10-05
m7a_conversion,2020-2025,4589,"engine_on_attempts: brier 0.2249, ECE 0.0272",2026-10-05
m7a_fg,2020-2025,6312,brier 0.1128; logloss 0.3632; ece 0.0113,2026-10-05
m7a_punt,2020-2025,12417,"crps 5.4610; crps_baseline 5.5661; crps_vs_baseline -0.1051 [-0.1258, -0.0863]",2026-10-05
m7a_wp,2020-2025,233913,brier 0.1559; logloss 0.4679; ece 0.0092,2026-10-05
m7a_wp_isotonic,2020-2025,233913,brier 0.1565; logloss 0.4746; ece 0.0066,2026-10-05


## 10. M7b: early-down play calling

Run versus pass by personnel, estimated with causal methods (propensity scores, doubly-robust AIPW, off-policy
evaluation) and checked with a placebo. **Passed**: the recommended policy beats observed play calling by
+0.052 EPA per play on the holdout, with a placebo that includes zero.

In [13]:
with section("10. M7b play calling"):
    recs_m7b = run_main("ml_m7b", ["dev"])
match_table(recs_m7b)
locked_holdout(["m7b_"])

frame through 2019: 78,564 early-down plays in 1,068 games (8s)


main pipeline:


  season 2018: 19,449 plays, train 39,508 (13s: cross_fit 6s, fit_predict 6s, cells_ope 1s)


  season 2019: 19,607 plays, train 58,957 (26s: cross_fit 18s, fit_predict 7s, cells_ope 1s)


placebo (actions shuffled within season x cell, whole pipeline rerun):


...


[10. M7b play calling: 75s]


**Retrained vs the committed development record**: 8 of 8 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
m7b_ope,ope_epa.diff,0.07428,0.07428,0.00000,0.00000,(none: exact),yes,OK
m7b_ope_thr10,ope_epa.diff,0.06509,0.06509,0.00000,0.00000,(none: exact),yes,OK
m7b_ope_trim10,ope_epa.diff,0.07427,0.07427,0.00000,0.00000,(none: exact),yes,OK
m7b_outcome,mse,1.46220,1.46220,0.00000,0.00000,(none: exact),yes,OK
m7b_placebo,ope_epa.diff,-0.00784,-0.00784,0.00000,0.00000,(none: exact),yes,OK
m7b_policy,recs.main.2018.cells,60.00000,60.00000,0.00000,0.00000,(none: exact),yes,OK
m7b_propensity,logloss,1.84867,1.84867,0.00000,0.00000,(none: exact),yes,OK
m7b_team_view,season,2019.00000,2019.00000,0.00000,0.00000,(none: exact),yes,OK


**Locked 2020-2025 holdout, loaded from `experiments/runs/` (not recomputed)**

record,seasons,n,result (diff = [95% CI]),committed
m7b_ope,2020-2025,122917,"ope_epa +0.0524 [+0.0377, +0.0670]; ope_succ +0.0407 [+0.0351, +0.0464]; share_policy_applies 0.6667; epa_per_affected_play 0.0785; observed_check_epa -0.0008 [-0.0014, -0.0002]; observed_check_succ -0.0005 [-0.0010, -0.0001]; placebo_epa +0.0058 [-0.0007, +0.0124]",2026-10-05
m7b_ope_thr10,2020-2025,122917,"ope_epa +0.0453 [+0.0334, +0.0569]; ope_succ +0.0343 [+0.0298, +0.0384]; share_policy_applies 0.5294",2026-10-05
m7b_ope_trim10,2020-2025,122917,"ope_epa +0.0538 [+0.0398, +0.0675]; ope_succ +0.0419 [+0.0367, +0.0471]; share_policy_applies 0.6898",2026-10-05
m7b_outcome,2020-2025,122917,"mse 1.3216; mse_cell_action 1.3342; mse_vs_cell_action -0.0125 [-0.0145, -0.0106]; mse_constant 1.3332; brier_succ 0.2393; brier_succ_cell_action 0.2419",2026-10-05
m7b_placebo,2020-2025,122917,"ope_epa +0.0058 [-0.0007, +0.0124]; ope_succ +0.0008 [-0.0022, +0.0040]; share_policy_applies 0.0954",2026-10-05
m7b_policy,2020-2025,122917,(descriptive tables only; see the record),2026-10-05
m7b_propensity,2020-2025,122917,"logloss 1.7609; logloss_cell_baseline 1.9367; logloss_vs_baseline -0.1758 [-0.1832, -0.1681]; pass_share_pred 0.5251; pass_share_obs 0.5266",2026-10-05


## 11. M8: game model v2

Margin and EPA targets, model blending, an opponent-adjusted QB value with injury-exit cleaning, early-season
(new QB, new coach) and context features (travel, rest, altitude, venue), and stacking. A **negative result**:
nothing beat C2d beyond noise on the development seasons, so A4s stays the headline and C2d the shadow. No holdout
was run.

In [14]:
with section("11. M8 game model v2"):
    recs_m8 = []
    if FULL:
        recs_m8 += run_main("ml_m8", ["tune"])
    recs_m8 += run_main("ml_m8", ["dev"])
match_table(recs_m8)
print("No holdout result exists for M8 by design: nothing beat C2d on the development seasons.")

== QB v2 tuning, 2000-2005: 3036 starter-games, k 100 ==
baseline (M3 value) wMSE 0.10918; v2 code path with nothing on 0.10918 (must match)


(a) opponent adjustment (prediction Q + D[opponent] + h*home; and Q alone):
 lam_d  wmse_opp  wmse_q_only
  25.0  0.115791     0.111649
  50.0  0.112501     0.110953
 100.0  0.109959     0.110393
 200.0  0.108448     0.110018
 400.0  0.107964     0.109809
 800.0  0.108168     0.109711
1600.0  0.108604     0.109672
3200.0  0.108985     0.109658


(b) injury-exit cleaning (partial QB-games: 17.8% of QB-game rows, 2000-2005):
 exit_weight   wmse_b  wmse_ab
        0.00 0.109071 0.108255
        0.25 0.109074 0.108119
        0.50 0.109095 0.108030
        0.75 0.109131 0.107980
        1.00 0.109182 0.107964
chosen: lam_d 400; exit_weight 0 (b alone), 1 (with a). 5.7s


== Kalman carry-over, REG weeks 1-4 of 2000-2005 (356 games) ==
TUNED gamma 0.4832, q_season 14.431: -loglik 1410.55
early-season optimum gamma 0.5010, q_season 23.106: -loglik 1409.37 (74 evaluations, 2.0s)
gamma profile (q_season fixed): 0.3 1413.04, 0.4 1411.36, 0.5 1410.46, 0.6 1410.41, 0.7 1411.28, 0.8 1413.10


== Blend weight, leave-one-season-out over REG 2001-2005 (n 1272), A4s features ==
  w  logloss  corr_target_margin
0.0 0.632534            0.850503
0.1 0.632281            0.875741
0.2 0.632063            0.899616
0.3 0.631882            0.921730
0.4 0.631742            0.941689
0.5 0.631644            0.959127
0.6 0.631592            0.973722
0.7 0.631588            0.985225
0.8 0.631635            0.993469
0.9 0.631736            0.998384
1.0 0.631893            1.000000
chosen w 0.7 (1.3s)

Freeze in code:
  qb.TUNED_V2_A = QBv2(lam_d=400)
  qb.TUNED_V2_B = QBv2(exit_weight=0)
  qb.TUNED_V2_AB = QBv2(lam_d=400, exit_weight=1)
  kalman.TUNED_CARRYOVER = replace(TUNED, gamma=0.50096, q_season=23.106)
  ml_m8.BLEND_W = 0.7


...

  E5:
    weeks_1_4    n  751 0.2233  vsA4s -0.0010 [-0.0029, +0.0009]  vsC2d -0.0004 [-0.0016, +0.0007]  (A4s 0.2243, C2d 0.2237, mkt 0.2198)
    weeks_5_9    n  969 0.2127  vsA4s +0.0002 [-0.0017, +0.0020]  vsC2d +0.0003 [-0.0001, +0.0008]  (A4s 0.2126, C2d 0.2124, mkt 0.2095)
    weeks_10_18  n 1730 0.2106  vsA4s -0.0019 [-0.0034, -0.0005]  vsC2d -0.0004 [-0.0010, +0.0002]  (A4s 0.2125, C2d 0.2109, mkt 0.2072)
    changed      n  664 0.2139  vsA4s -0.0010 [-0.0034, +0.0013]  vsC2d -0.0010 [-0.0022, +0.0001]  (A4s 0.2150, C2d 0.2150, mkt 0.2072)
    unchanged    n 2786 0.2139  vsA4s -0.0012 [-0.0023, -0.0000]  vsC2d +0.0000 [-0.0004, +0.0004]  (A4s 0.2151, C2d 0.2139, mkt 0.2114)
    short_week   n  179 0.1991  vsA4s -0.0050 [-0.0092, -0.0007]  vsC2d -0.0012 [-0.0027, +0.0002]  (A4s 0.2041, C2d 0.2003, mkt 0.1893)
  E5m:
    weeks_1_4    n  751 0.2233  vsA4s -0.0010 [-0.0026, +0.0006]  vsC2d -0.0004 [-0.0015, +0.0006]  (A4s 0.2243, C2d 0.2237, mkt 0.2198)
    weeks_5_9    n  969 0.21


32 runs written to data/raw/ml/_train_all_runs/

[11. M8 game model v2: 33s]


**Retrained vs the committed development record**: 35 of 35 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
m8_tune_qb_v2,wmse_base,0.10918,0.10918,0.00000,0.00000,(none: exact),yes,OK
m8_tune_carryover,neg_loglik,1409.37308,1409.37308,0.00000,0.00000,(none: exact),yes,OK
m8_tune_blend,logloss,0.63159,0.63159,0.00000,0.00000,(none: exact),yes,OK
m8_A4s,brier,0.21509,0.21509,0.00000,0.00000,(none: exact),yes,OK
m8_C2d,brier,0.21413,0.21413,0.00000,0.00000,(none: exact),yes,OK
m8_E1_A4s_epa_logit,brier,0.21541,0.21541,-0.00000,-0.00000,ece_null_p95,yes,OK
m8_E1_A4s_epa_normal,brier,0.21559,0.21559,0.00000,0.00000,(none: exact),yes,OK
m8_E1_A4s_margin_logit,brier,0.21496,0.21496,0.00000,0.00000,ece,yes,OK
m8_E1_A4s_margin_normal,brier,0.21512,0.21512,0.00000,0.00000,(none: exact),yes,OK
m8_E1_A4s_blend_logit,brier,0.21498,0.21498,0.00000,0.00000,(none: exact),yes,OK


No holdout result exists for M8 by design: nothing beat C2d on the development seasons.


## 12. The 2026 production models

Fits the models that run live this season, the way `scripts/ml_predict.py` does, on every completed season through
2025: **A4s** (the headline win probability), the **margin model**, and the **C2d shadow model**. Each fit first
re-fits the last development season and stops unless it reproduces the committed coefficients. Then it predicts every
unplayed 2026 game and simulates the rest of the season for playoff odds. **Nothing is written**: not the ledger, not
`experiments/live/`, not `site/data/`.

In [15]:
import ml_predict as mp
import export_site as xs
from nflelo.ml import live, live_features
from nflelo.ml.sim import season as sim_season

with section("12a. 2026 models: fit"):
    # ml_predict reads Elo from outputs/ (written by build.py). Point it at the in-memory rebuild instead.
    elo_out = ELO.drop(columns=["source"]).copy()
    for c in ["home_pre", "away_pre", "home_post", "away_post"]:
        elo_out[c] = elo_out[c].round(2)
    xs.load_elo = lambda: elo_out
    import build as build_script
    RATINGS = build_script.current_ratings(ELO, config.DEFAULT_CONFIG, ELO_STATE)
    xs.load_ratings = lambda: RATINGS

    NOW = pd.Timestamp.now(tz="UTC").floor("s")
    pbp26, sched26, games26 = mp.load(CURRENT)               # games26 = the in-memory rebuild
    mp.check_elo_fresh(sched26, games26, elo_out, CURRENT)   # Elo includes every finished game
    MODEL = mp.fit_model(CURRENT, pbp26, sched26, games26)
    MARGIN = mp.fit_margin_model(CURRENT, pbp26, sched26, games26)
    SHADOW = mp.fit_shadow_model(CURRENT, pbp26, sched26, games26)
    VERSION = mp.combined_version(MODEL, MARGIN)

coef = pd.DataFrame({"A4s": {"intercept": MODEL["intercept"], **MODEL["coef"]},
                     "C2d shadow": {"intercept": SHADOW["intercept"], **SHADOW["coef"]}})
display(coef.round(4))
print(f"A4s {MODEL['model_version']}: trained on REG {MODEL['train']['seasons'][0]}-{MODEL['train']['seasons'][1]} "
      f"(n = {MODEL['train']['n_games']})")
print(f"margin {MARGIN['version']}: sigma {MARGIN['fit']['sigma']:.2f}, shape {MARGIN['shape']}")

reproduction check: 2025 coefficients match experiments/runs/20261003T215950Z_m3_signoff_A4s.json (max |diff| 6.7e-16)


margin reproduction check: 2019 fit matches experiments/runs/20261005T021003Z_m4_margin_keynum.json (max |diff| 2.8e-14)


shadow reproduction check: 2019 C2d coefficients match experiments/runs/20261005T062809Z_m3b_C2d.json (max |diff| 1.7e-16)

[12a. 2026 models: fit: 12s]


,A4s,C2d shadow
intercept,0.0094,-0.0538
elo_logit,0.7967,0.5336
adj_epa_margin,1.1489,NaN
qb_delta_diff,3.6999,3.6253
kf_margin,NaN,0.0643


A4s A4s-2026-e25b31e2: trained on REG 2001-2025 (n = 6471)
margin M-2026-4d6cbdab: sigma 13.26, shape keynum


In [16]:
with section("12b. 2026 models: predict + simulate"):
    pred = mp.predict(CURRENT, NOW, pbp26, sched26, games26, MODEL, MARGIN)
    sp = mp.shadow_predict(SHADOW, pred, pbp26, sched26, CURRENT, NOW)
    nxt = live_features.next_week(sched26[sched26["season"] == CURRENT], NOW)

    SIM_DIR = CAPTURE / "sim"
    SIM_DIR.mkdir(exist_ok=True)
    n_sims = sim_season.DEFAULT_SIMS if FULL else 4000
    odds = mp.run_sim(CURRENT, NOW, pbp26, sched26, games26, MARGIN, VERSION, SIM_DIR, n_sims)

view = pred.join(sp[["p_home_shadow", "kf_margin"]])
wk_last = nxt if nxt is not None else int(view["game_week"].max())
view = view[view["game_week"] <= wk_last].sort_values("kickoff_utc")     # rest of the current week plus the next slate
tbl = pd.DataFrame({
    "week": view["game_week"], "kickoff (ET)": view["kickoff_utc"].dt.tz_convert("America/New_York").dt.strftime("%a %b %d %H:%M"),
    "game": view["away_team"] + " @ " + view["home_team"],
    "NFLELO model: home win %": (100 * view["p_home_model"]).round(1),
    "shadow (C2d) %": (100 * view["p_home_shadow"]).round(1),
    "Elo %": (100 * view["p_home_elo"]).round(1),
    "Vegas % (benchmark)": (100 * view["p_home_market"]).round(1),
    "model spread (home)": view["spread_model"].round(1),
    "QB news": view["qb_delta_diff"].round(3),
})
display(Markdown(f"**This week's predictions** (as of {NOW:%Y-%m-%d %H:%M} UTC; {len(pred)} unplayed regular-season games; "
                 f"showing weeks {int(view['game_week'].min())}-{wk_last}). Vegas is a benchmark only and never an input."))
display(tbl.reset_index(drop=True).style.hide(axis="index"))

simulation: 20,000 seasons, 180 games left, tau_rest 4.5, seed 1792470396; state 0.9s, simulate 3.9s (draws 0.2s, tiebreakers 3.6s, bracket 0.0s); appended 32 rows to /Users/walker_tracy/Desktop/Old Projects/NFLELO/data/raw/ml/_train_all_runs/sim/sim_2026.csv
Playoff odds, top 5:    SF 97% (SB 15.3%); MIN 91% (SB 8.5%); BUF 90% (SB 10.1%); KC 88% (SB 7.0%); JAX 86% (SB 8.0%)
Playoff odds, bottom 5: NYJ 9% (SB 0.2%); ARI 6% (SB 0.2%); TEN 1% (SB 0.0%); MIA 1% (SB 0.0%); LAC 1% (SB 0.0%)

[12b. 2026 models: predict + simulate: 6s]


**This week's predictions** (as of 2026-10-20 04:26 UTC; 180 unplayed regular-season games; showing weeks 6-7). Vegas is a benchmark only and never an input.

week,kickoff (ET),game,NFLELO model: home win %,shadow (C2d) %,Elo %,Vegas % (benchmark),model spread (home),QB news
6,Mon Oct 19 20:15,WAS @ SF,80.600000,80.100000,82.000000,76.600000,10.900000,-0.010000
7,Thu Oct 22 20:15,NE @ CHI,53.600000,53.600000,49.300000,58.300000,1.300000,0.045000
7,Sun Oct 25 09:30,PIT @ NO,36.000000,35.800000,33.300000,37.800000,-4.000000,0.013000
7,Sun Oct 25 13:00,SF @ ATL,27.000000,28.000000,29.300000,nan,-7.100000,-0.036000
7,Sun Oct 25 13:00,CIN @ BAL,45.200000,45.000000,58.900000,57.200000,-1.000000,-0.139000
7,Sun Oct 25 13:00,TB @ CAR,65.200000,65.800000,65.400000,62.200000,4.900000,0.020000
7,Sun Oct 25 13:00,NYG @ HOU,72.800000,72.900000,68.100000,79.600000,7.600000,0.053000
7,Sun Oct 25 13:00,IND @ MIN,77.200000,75.400000,79.400000,70.500000,9.200000,0.005000
7,Sun Oct 25 13:00,MIA @ NYJ,61.700000,62.100000,57.300000,66.900000,3.800000,0.038000
7,Sun Oct 25 13:00,CLE @ TEN,32.900000,37.000000,36.800000,46.800000,-5.100000,-0.023000


In [17]:
o = odds.sort_values(["playoffs", "win_sb"], ascending=False)
cols = {"wins_mean": "proj. wins", "playoffs": "make playoffs %", "win_div": "win division %", "seed1": "top seed %", "win_sb": "win title %"}
t = o[[c for c in cols if c in o.columns]].rename(columns=cols)
for c in t.columns:
    if "%" in c:
        t[c] = (100 * t[c]).round(1)
t["proj. wins"] = t["proj. wins"].round(1)
display(Markdown(f"**Playoff odds**: {n_sims:,} simulated seasons from the current standings and ratings "
                 f"(model {VERSION})."))
display(t.head(12))

**Playoff odds**: 20,000 simulated seasons from the current standings and ratings (model A4s-2026-e25b31e2+M-2026-4d6cbdab).

,proj. wins,make playoffs %,top seed %,win title %
team,,,,
SF,13.2,96.6,50.9,15.3
MIN,12.1,90.9,28.5,8.5
BUF,11.9,90.0,26.2,10.1
KC,11.6,87.6,24.0,7.0
JAX,11.3,86.4,13.3,8.0
NE,10.5,77.1,10.3,6.2
DEN,10.7,75.4,11.4,5.7
SEA,10.3,72.0,5.0,8.6
CHI,10.2,67.3,4.6,5.0


## 13. Timing and integrity

Wall-clock time per section, a roll-up of every retrained development run against the committed record, and the
checks that this notebook changed nothing tracked.

In [18]:
total = sum(SECTION_SECONDS.values())
timing = pd.DataFrame({"section": list(SECTION_SECONDS), "seconds": [round(v, 1) for v in SECTION_SECONDS.values()],
                       "minutes": [round(v / 60, 1) for v in SECTION_SECONDS.values()]})
timing.loc[len(timing)] = ["TOTAL (sections)", round(total, 1), round(total / 60, 1)]
display(Markdown(f"**Timing, mode `{MODE}`**"))
display(timing.style.hide(axis="index"))

**Timing, mode `full`**

section,seconds,minutes
1. Data refresh,4.100000,0.100000
2. Elo v2 + baselines,0.200000,0.000000
3a. M3 game model ladder,37.500000,0.600000
3b. M3 no-Elo study,11.700000,0.200000
4. M3b Kalman (C2d),226.700000,3.800000
5. M4 margin + playoff odds,1099.600000,18.300000
6. M5 box-score player values,167.200000,2.800000
7. M5b RAPM,102.700000,1.700000
8. M6 play model,407.500000,6.800000
9. M7a win prob + fourth down,2208.700000,36.800000


In [19]:
# Roll-up: every run this notebook retrained that has a committed development record.
summary = match_table(ALL_RECS, title="All retrained development runs vs the committed record")
bad = summary[~summary["status"].isin(["OK"])]
if len(bad):
    display(Markdown("**Runs that differ from, or have no, committed record:**"))
    display(bad)

**All retrained development runs vs the committed record**: 126 of 126 runs match (tolerance max(5e-05, 0.2% of the record) on every metric).

run,key metric,record,retrained,diff,largest diff,on metric,same games,status
coin_flip,brier,0.24935,0.24935,0.00000,0.00000,(none: exact),yes,OK
elo_v2,brier,0.21782,0.21782,0.00000,0.00000,(none: exact),yes,OK
home_rate,brier,0.24514,0.24514,0.00000,0.00000,(none: exact),yes,OK
market,brier,0.21060,0.21060,0.00000,0.00000,(none: exact),yes,OK
m3_tune_ratings,mse,0.13043,0.13043,0.00000,0.00000,(none: exact),yes,OK
m3_A0,brier,0.21783,0.21783,0.00000,0.00000,(none: exact),yes,OK
m3_A1,brier,0.21796,0.21796,0.00000,0.00000,(none: exact),yes,OK
m3_A2,brier,0.21765,0.21765,0.00000,0.00000,(none: exact),yes,OK
m3_A3,brier,0.21802,0.21802,0.00000,0.00000,(none: exact),yes,OK
m3_A4,brier,0.21500,0.21500,0.00000,0.00000,(none: exact),yes,OK


In [20]:
# Integrity checks: nothing tracked changed, the registry is untouched, no run touched the locked seasons.
new_changes = git_dirty() - GIT_AT_START
allowed = {"notebooks/train_all_models.ipynb", "scripts/train_all.sh", ".gitignore"}
unexpected = sorted(l for l in new_changes if l[3:] not in allowed)
runs_now = sorted(p.name for p in REAL_RUNS.glob("*.json"))
assert runs_now == RUNS_AT_START, "the committed run registry changed"
assert not unexpected, f"unexpected git changes: {unexpected}"
assert not any(r.get("holdout") for r in ALL_RECS), "a retrained run touched the locked holdout seasons"
print(f"registry files: {len(runs_now)} (unchanged)")
print(f"retrained run records captured outside the registry: {len(list(CAPTURE.glob('*.json')))} in {CAPTURE.relative_to(ROOT)}/")
print("new git changes from this run:", sorted(new_changes) or "none")
print("holdout seasons touched by any retrained run: none")

registry files: 167 (unchanged)
retrained run records captured outside the registry: 126 in data/raw/ml/_train_all_runs/
new git changes from this run: none
holdout seasons touched by any retrained run: none
